# Entrenamiento y evaluación reproducibles

Este notebook reproduce la preparación, comparación y evaluación de los dos sistemas del trabajo: regresión del consumo de las próximas 24 horas y clasificación de exceso. La implementación detallada está en [`entrenamiento_modelos.py`](../entrenamiento_modelos.py); el notebook ejecuta ese flujo y presenta sus resultados.

**Ejecución:** abrir desde la raíz del repositorio y ejecutar todas las celdas. Requiere las dependencias de `requirements.txt` y conexión a Internet la primera vez para descargar el archivo público de UCI (20 MB). El ZIP se guarda en la caché del usuario, no en el repositorio. La evaluación no modifica los modelos usados por la aplicación.

## Pregunta, filas y objetivos

La fuente es UCI, *Individual Household Electric Power Consumption*: mediciones por minuto de un hogar de Sceaux, Francia (2006–2010). Cada fila supervisada representa una hora de pronóstico. Los predictores describen la hora disponible y el historial de consumo; el objetivo de regresión es la energía activa observada en las **24 horas siguientes**. La clase positiva significa que ese objetivo supera **25,917658 kWh**. Se mantiene el umbral que usa la aplicación y se fija antes de evaluar; no se optimiza mirando el conjunto de prueba.

La temperatura diaria no es un pronóstico. Para no filtrar observaciones meteorológicas futuras, `temp_actual` y `temp_next24` usan como aproximación la temperatura media del último día completo disponible (persistencia). La aplicación permite ingresar sus propios valores; los resultados del notebook no eliminan la limitación de transferir datos de un hogar francés a viviendas argentinas.

## Preparación, calidad y fuga de información

- Se descargan solo las columnas de fecha/hora, potencia activa/reactiva, voltaje y tres submedidores. `?` se convierte en faltante.
- Se auditan timestamps inválidos, filas duplicadas exactas, faltantes y valores físicos imposibles (potencias/reactiva/submedidores negativas, potencia activa mayor a 15 kW o voltaje no positivo). Las mediciones imposibles se excluyen de la agregación; no se imputan los faltantes.
- Se agregan horas con al menos 45 de sus 60 minutos válidos en **cada** medición. La potencia activa/reactiva se promedia; los submedidores, registrados en Wh por minuto, se escalan al equivalente de 60 minutos.
- Los predictores de consumo son ventana actual/previa de 24 horas, consumo equivalente de la semana anterior, cos φ, submedidores y voltaje; los demás son calendario y temperatura con persistencia. El objetivo usa exclusivamente las 24 horas posteriores al instante de predicción.
- La división es temporal y ordenada: aproximadamente 70 % entrenamiento, 15 % validación y 15 % prueba. Se purgan 24 horas en los límites para que las ventanas de objetivos no se solapen. Los mismos instantes de validación y prueba se usan para todos los candidatos y para comparar las alertas.
- `StandardScaler` se ajusta dentro del `Pipeline` KNN, por lo que aprende únicamente del entrenamiento. Validación selecciona candidatos; prueba se reserva para la evaluación final y no se usa para escogerlos.

El umbral de 25,917658 kWh es la referencia existente de la aplicación, no una recomendación universal para hogares. Un falso negativo omite una advertencia de exceso y puede dejar menos tiempo para reaccionar; se le asigna costo relativo 3. Un falso positivo genera una advertencia innecesaria; se le asigna costo 1. Es un criterio explícito de decisión, no una valoración monetaria.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "entrenamiento_modelos.py").is_file():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Abrí el notebook desde una carpeta dentro del repositorio.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display
from entrenamiento_modelos import run_experiment

resultado = run_experiment()
print("Regresor elegido por validación:", resultado["best_regression"])
print("Clasificador elegido por costo de validación:", resultado["best_classifier"])

In [ ]:
print("Auditoría del dataset original")
display(pd.Series(resultado["raw_audit"], name="valor").to_frame())
print("Preparación y objetivo")
display(pd.Series(resultado["sample_audit"], name="valor").to_frame())
print("Particiones cronológicas")
display(resultado["split_sizes"])

## Regresión: baselines y candidatos

Se compara `DummyRegressor` con árboles de decisión de profundidad 3, 6, 12 y sin límite, y KNN con k=5, 25 y 50. MAE y RMSE están expresados en kWh; R² se interpreta como proporción de variabilidad explicada, no como probabilidad de acierto. Los residuos se calculan como consumo real menos estimado.

In [ ]:
print("Validación — ordenada por MAE")
display(resultado["regression_validation"].round(3))
print("Prueba — mismos candidatos, nunca usada para seleccionarlos")
display(resultado["regression_test"].round(3))
print("Errores absolutos más grandes del regresor elegido en validación")
display(resultado["error_cases"].round(3))

## Clasificación: baselines y candidatos

Se compara `DummyClassifier` con árboles (profundidades 3, 6, 12 y sin límite), KNN (k=5, 25 y 50) y `GaussianNB`. Se informa la matriz de confusión en sus cuatro componentes (TN, FP, FN, TP), además de accuracy, precision, recall y F1 de la clase exceso. La selección en validación prioriza el costo relativo definido en Preparación; en caso de empate, prioriza F1.

In [ ]:
print("Validación — ordenada por costo")
display(resultado["classification_validation"].round(3))
print("Prueba — evaluación final")
display(resultado["classification_test"].round(3))

## Comparación integrada y errores de decisión

La alerta de regresión se obtiene comparando la predicción con el umbral; el clasificador produce su clase directamente. Se comparan ambas sobre exactamente las mismas filas de prueba. Los falsos positivos y negativos se acompañan de consumo real, predicción y distancia al umbral para identificar errores cercanos al límite.

In [ ]:
display(resultado["integrated_test"].round(3))
print(f"Acuerdo entre ambas alertas: {resultado['alert_agreement']:.1%}")
print("Casos reales más cercanos al umbral")
display(resultado["near_threshold"].round(3))
print("Falsos negativos del clasificador directo elegido en validación")
display(resultado["false_negatives"].round(3))
print("Falsos positivos del clasificador directo elegido en validación")
display(resultado["false_positives"].round(3))

## Conclusiones y uso responsable

Interpretar las métricas junto con el período temporal: los patrones cambian entre entrenamiento, validación y prueba. Reportar todos los resultados, incluso cuando un candidato no supera el baseline. No escoger el sistema por el conjunto de prueba.

El costo menor no convierte al predictor en una garantía ni hace apropiado usarlo para decisiones automáticas: los datos representan un solo hogar, el umbral es una referencia del proyecto y la distribución de excesos puede variar por estación. La interfaz debe comunicar tanto las respuestas como la incertidumbre, y la recomendación final debe priorizar que la persona pueda revisar sus consumos antes de asumir un gasto inesperado.